<a href="https://colab.research.google.com/github/samsure13003/llm_fine_tune/blob/main/LLM_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LLM Fine Tuning

In [1]:
!pip install -q transformers trl peft datasets accelerate evaluate rouge_score huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00


In [8]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 108.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [5]:
from datasets import load_dataset

raw = load_dataset("yahma/alpaca-cleaned", split="train[:2500]")
split = raw.train_test_split(test_size=200, seed=42)
train_raw, test_raw = split["train"], split["test"]   # test_raw is NEVER used for training
print(train_raw[0])   # look at one example: instruction, input, output

{'output': '100 grams is equivalent to approximately 0.22046 pounds.', 'input': '', 'instruction': 'Convert 100 grams to pounds.'}


In [6]:
def to_chat(ex):
    q = ex["instruction"] + ("\n" + ex["input"] if ex["input"] else "")
    return {"messages": [{"role": "user", "content": q},
                         {"role": "assistant", "content": ex["output"]}]}

train_ds = train_raw.map(to_chat, remove_columns=train_raw.column_names)

Map:   0%|          | 0/2300 [00:00<?, ? examples/s]

In [10]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

trainer = SFTTrainer(
    model=MODEL,
    train_dataset=train_ds,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="out",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        max_length=512,
        remove_unused_columns=False,
        fp16=True, # Force FP16 precision for older GPUs like T4
        bf16=False
    ),
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules="all-linear",
        task_type="CAUSAL_LM"
    ),
)
trainer.train()
trainer.save_model("my-lora")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
10,1.455120
20,1.216232
30,1.224651
40,1.177043
50,1.223967
60,1.237973
70,1.173437
80,1.183500
90,1.145058
100,1.238927


In [13]:
trainer.model.push_to_hub("samsur13/qwen-alpaca-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 38.0kB / 35.2MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/samsur13/qwen-alpaca-lora/commit/ca24a88300a3c687ff32d3921d49c98aa2a58286', commit_message='Upload model', commit_description='', oid='ca24a88300a3c687ff32d3921d49c98aa2a58286', pr_url=None, repo_url=RepoUrl('https://huggingface.co/samsur13/qwen-alpaca-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='samsur13/qwen-alpaca-lora'), pr_revision=None, pr_num=None)

In [14]:
import torch, evaluate
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

prompts   = [e["instruction"] + ("\n" + e["input"] if e["input"] else "") for e in test_raw.select(range(50))]
refs      = [e["output"] for e in test_raw.select(range(50))]

def ask(model, tok, prompts, max_new_tokens=150):
    outs = []
    for p in prompts:
        text = tok.apply_chat_template([{"role": "user", "content": p}],
                                       tokenize=False, add_generation_prompt=True)
        ids = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            g = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
        outs.append(tok.decode(g[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
    return outs

# A and B share one model: adapter OFF = base, adapter ON = fine-tuned
tok  = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
tuned = PeftModel.from_pretrained(base, "my-lora")

out_B = ask(tuned, tok, prompts)
with tuned.disable_adapter():
    out_A = ask(tuned, tok, prompts)

# C: bigger rival
BIG = "Qwen/Qwen2.5-1.5B-Instruct"
tok_c = AutoTokenizer.from_pretrained(BIG)
big = AutoModelForCausalLM.from_pretrained(BIG, torch_dtype=torch.float16, device_map="auto")
out_C = ask(big, tok_c, prompts)

# Score with ROUGE-L (how much each answer overlaps with the reference answer)
rouge = evaluate.load("rouge")
for name, out in [("Base 0.5B", out_A), ("Fine-tuned 0.5B", out_B), ("Rival 1.5B", out_C)]:
    print(name, round(rouge.compute(predictions=out, references=refs)["rougeL"], 4))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base 0.5B 0.3151
Fine-tuned 0.5B 0.335
Rival 1.5B 0.3482
